# Variance reduction for small-dataset low-data DPO learning curves

**Question.** The small-dataset low-data DPO curves in
`report/meetings/14-07-26.ipynb` (C05 **ED1**, AbAgym **Cetuximab-H**) are
dominated by *test-set* noise: each point is a Spearman on a single fixed
held-out split of only ~44–56 sequences, so the seed-to-seed band is wide and
the low-N "evo-vs-vanilla" ordering is essentially a re-measurement of a noisy
zero-shot anchor (see that notebook's Check 2 / Conclusions). **Can we cut that
variance without collecting more data?**

This notebook studies variance-reduction methods for those curves. It is
organised as **baseline first, then one method section per technique**, so more
methods can be added later without disturbing the comparison:

- **Section A — original non-CV baseline.** Reproduces the 14-07-26 non-CV
  low-data tests (single fixed split) as the reference. *Kept visually separate
  from every CV result.*
- **Section B — CV setup & preflight.** Explains **k-fold cross-validation** and
  prints the exact cluster commands to generate any missing CV artifacts.
- **Section C — CV learning curves.** Low-data curves using the **pooled
  out-of-fold Spearman** (concatenate held-out predictions from all *k* folds,
  compute **one** Spearman over the full dataset) as the y-axis.
- **Section D — comparison / interpretation.** Baseline vs pooled-CV, emphasis
  on stability / variance reduction (separate figures, never one merged plot).
- **Section E — diagnostics.** Rows per fold, pooled counts, fold completion,
  foldwise Spearmans, seed-to-seed variance.

**The headline CV metric is the pooled OOF Spearman — NOT the mean of per-fold
Spearmans.** Each original row is held out for test exactly once across the *k*
folds, the held-out fold never leaks into train/val, and fold assignment is
deterministic in `(n_folds, fold_seed)`. All of that lives in the reusable
library `protein_design.cv_splitting` (fold-key `<base>_cv<K>s<SEED>_f<I>`),
not in this notebook.

*Analysis-only notebook: it reads already-produced artifacts from cluster
storage and, when something is missing, prints the exact `sbatch` / `uv run`
command to build it — it never submits jobs itself.*

In [1]:
# === Setup: cluster paths, registry helpers, shared scan/plot machinery ======
%matplotlib inline
import os, re, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.transforms import blended_transform_factory, ScaledTranslation
import yaml
# scipy.stats is only needed where we (re)compute Spearman/bootstrap CIs.
from scipy.stats import spearmanr

DATE = "0717"  # figure-name suffix (2026-07-17)


# --- REPO_ROOT: walk upward until conf/analysis/models.yaml is found ---------
def _find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "conf" / "analysis" / "models.yaml").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root (conf/analysis/models.yaml) above cwd.")

REPO_ROOT = _find_repo_root()
USER = os.environ.get("USER", "unknown")

# --- Cluster storage conventions (env-overridable) ---------------------------
TRAIN_DIR = Path(os.environ.get(
    "TRAIN_DIR",
    str(Path(os.environ.get("SCRATCH_DIR", f"/cluster/scratch/{USER}/protein-design")) / "train"),
))
ANALYSIS_DIR = Path(os.environ.get(
    "ANALYSIS_DIR", "/cluster/project/infk/krause/mdenegri/protein-design/analysis"))
# Raw datasets / canonical (non-CV) split CSVs live on shared project storage.
DATA_DIR = Path("/cluster/project/infk/krause/mdenegri/protein-design/data")
# CV fold splits are written per-user by scripts/data_prep/build_cv_splits.py.
CV_SPLIT_DIR = Path(os.environ.get(
    "CV_SPLIT_DIR", f"/cluster/project/infk/krause/{USER}/protein-design/data/dms_splits_cv"))

FIG_DIR = REPO_ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT   :", REPO_ROOT)
print("TRAIN_DIR   :", TRAIN_DIR, "(exists)" if TRAIN_DIR.exists() else "(MISSING)")
print("ANALYSIS_DIR:", ANALYSIS_DIR, "(exists)" if ANALYSIS_DIR.exists() else "(MISSING)")
print("CV_SPLIT_DIR:", CV_SPLIT_DIR, "(exists)" if CV_SPLIT_DIR.exists() else "(MISSING)")

# --- Model registry (single source of truth for labels/colors) --------------
_REG = yaml.safe_load((REPO_ROOT / "conf" / "analysis" / "models.yaml").read_text()).get("models", {})
def lc_label(key):
    return _REG.get(key, {}).get("label") or key
def lc_color(key, i=0):
    c = _REG.get(key, {}).get("color")
    cyc = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    return c if c else cyc[i % len(cyc)]

# --- Experiment scope (same as the 14-07-26 generalization notebook) ---------
MODELS = ["vanilla_35m", "evo_35m"]
SEEDS = [0, 1, 2]                      # low_data seeds (error bars)
N_FOLDS = 5                            # k-fold CV
FOLD_SEED = 0
_MODELS_RE = "|".join(re.escape(m) for m in MODELS)

# Per-dataset config: N grid, canonical split dir, ground-truth metric column,
# total train size (for the % secondary axis), and cetuximab's extra overrides.
DATASETS = {
    "ed1_m22": {
        "test_set": "ed1",
        "n_grid": [20, 50, 100, 200, 366],
        "split_dir": DATA_DIR / "dms_splits" / "ed1_m22",
        "metric_col": "M22_binding_enrichment_adj",
        "total_train": 366,
        "extra": "",
    },
    "cetuximab_h": {
        "test_set": "cetuximab_h",
        "n_grid": [20, 50, 100, 200, 447],
        "split_dir": DATA_DIR / "dms_splits" / "cetuximab_h",
        "metric_col": "neg_DMS_score",
        "total_train": 447,
        "extra": (" data.delta_based.strong_pos_threshold=2.5"
                  " data.delta_based.strong_neg_threshold=-0.25 model.use_context=false"),
    },
}

_METRIC = "test_spearman_avg"
_TS_RE = re.compile(r"_\d{8}_\d{6}$")
print("\nScope:", MODELS, "| seeds", SEEDS, "| k-fold", N_FOLDS, "seed", FOLD_SEED)

REPO_ROOT   : /cluster/home/gguidarini/protein-design
TRAIN_DIR   : /cluster/scratch/gguidarini/protein-design/train (exists)
ANALYSIS_DIR: /cluster/project/infk/krause/mdenegri/protein-design/analysis (exists)
CV_SPLIT_DIR: /cluster/project/infk/krause/gguidarini/protein-design/data/dms_splits_cv (exists)

Scope: ['vanilla_35m', 'evo_35m'] | seeds [0, 1, 2] | k-fold 5 seed 0


## Section A — original non-CV baseline (reference)

Reproduces the 14-07-26 low-data tests **unchanged**: one fixed 80/10/10 split
per dataset, `test_spearman_avg` read from each finished run's `summary.json`,
mean ± std across the 3 low-data seeds per `(model, N)`. This is the reference
the CV method must beat on *stability*. It is deliberately kept in its own
section with its own plots — CV is introduced as a separate method below, never
as a silent replacement.

In [2]:
# === A0: shared run-scanning machinery (reused by baseline AND CV) ===========
# Same logic as report/meetings/14-07-26.ipynb: match finished
# lowdata_<model>_n<N>_s<seed>[...] run dirs, keep the most-recent per logical
# key, and disambiguate by data.test.dataset_key from resolved_config.yaml (the
# N grid overlaps other sweeps, so run-name matching alone is not enough).
def _run_test_dataset_key(run_dir: Path):
    try:
        c = yaml.safe_load((run_dir / "resolved_config.yaml").read_text())
        return (((c or {}).get("data", {}) or {}).get("test", {}) or {}).get("dataset_key")
    except Exception:
        return None

# Baseline (non-CV) runs: trailing (?:_.*)? tolerates suffixes but we EXCLUDE
# CV fold runs (..._cv<K>s<S>_f<I>) so the two analyses never mix.
_BASE_RE = re.compile(rf"^lowdata_(?P<model>{_MODELS_RE})_n(?P<n>\d+)_s(?P<seed>\d+)(?:_.*)?$")
_CV_SUFFIX_RE = re.compile(r"_cv\d+s\d+_f\d+$")

def _finished_noncv(run_name: str, target_dataset_key: str) -> bool:
    pat = re.compile(r"^" + re.escape(run_name) + r"_\d{8}_\d{6}$")
    for p in TRAIN_DIR.glob(run_name + "_*"):
        if not pat.match(p.name):
            continue
        if not (p / "summary.json").exists():
            continue
        if _run_test_dataset_key(p) == target_dataset_key:
            return True
    return False


def scan_baseline_runs(train_dir=TRAIN_DIR, metric=_METRIC) -> pd.DataFrame:
    """Finished non-CV lowdata runs -> tidy (model, n_train, seed, test_set, metric, run_dir)."""
    cols = ["model", "n_train", "seed", "test_set", metric, "run_dir"]
    if not train_dir.exists():
        return pd.DataFrame(columns=cols)
    best = {}
    for p in sorted(train_dir.glob("lowdata_*_n*_s*")):
        name = _TS_RE.sub("", p.name)
        if _CV_SUFFIX_RE.search(name):
            continue  # a CV fold run, handled in Section C
        m = _BASE_RE.match(name)
        if not m or not (p / "summary.json").exists():
            continue
        dkey = _run_test_dataset_key(p)
        ts = str(dkey).replace("_m22", "") if dkey else "unknown"
        key = (m.group("model"), int(m.group("n")), int(m.group("seed")), ts)
        try:
            mt = (p / "summary.json").stat().st_mtime
        except OSError:
            continue
        if key in best and best[key][0] >= mt:
            continue
        try:
            val = json.loads((p / "summary.json").read_text()).get(metric)
        except Exception:
            val = None
        best[key] = (mt, str(p), val)
    rows = [{"model": k[0], "n_train": k[1], "seed": k[2], "test_set": k[3],
             metric: v[2], "run_dir": v[1]} for k, v in best.items()]
    if not rows:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows).sort_values(["test_set", "model", "n_train", "seed"]).reset_index(drop=True)


baseline_df = scan_baseline_runs()
print(f"Baseline (non-CV) runs found: {len(baseline_df)}")
if not baseline_df.empty:
    print("Test sets:", sorted(baseline_df["test_set"].unique()))

Baseline (non-CV) runs found: 0


In [3]:
# === A1: baseline preflight — what exists, what's missing, how to build it ===
print("# --- non-CV baseline low-data sweeps (one launcher call per dataset) ---")
for key, spec in DATASETS.items():
    ng = ",".join(str(n) for n in spec["n_grid"])
    print(f"bash_scripts/dpo_lowdata_sweep.sh --models {','.join(MODELS)} "
          f"--n {ng} --seeds {','.join(map(str, SEEDS))} --model-preset esm2_35m "
          f"--task lora_dpo data.dpo_dataset_key={key} data.test.dataset_key={key}{spec['extra']}")

missing = []
for key, spec in DATASETS.items():
    for model in MODELS:
        for n in spec["n_grid"]:
            for s in SEEDS:
                rn = f"lowdata_{model}_n{n}_s{s}"
                if not _finished_noncv(rn, key):
                    missing.append(f"{rn}  [{key}]")

n_expected = sum(len(MODELS) * len(s["n_grid"]) * len(SEEDS) for s in DATASETS.values())
print(f"\nTRAIN_DIR = {TRAIN_DIR}")
if missing:
    print(f"{len(missing)} of {n_expected} baseline runs missing — run the launcher(s) above. First few:")
    for m in missing[:12]:
        print("  ", m)
    if len(missing) > 12:
        print(f"  ... and {len(missing) - 12} more")
else:
    print(f"All {n_expected} baseline runs present.")

# --- non-CV baseline low-data sweeps (one launcher call per dataset) ---
bash_scripts/dpo_lowdata_sweep.sh --models vanilla_35m,evo_35m --n 20,50,100,200,366 --seeds 0,1,2 --model-preset esm2_35m --task lora_dpo data.dpo_dataset_key=ed1_m22 data.test.dataset_key=ed1_m22
bash_scripts/dpo_lowdata_sweep.sh --models vanilla_35m,evo_35m --n 20,50,100,200,447 --seeds 0,1,2 --model-preset esm2_35m --task lora_dpo data.dpo_dataset_key=cetuximab_h data.test.dataset_key=cetuximab_h data.delta_based.strong_pos_threshold=2.5 data.delta_based.strong_neg_threshold=-0.25 model.use_context=false

TRAIN_DIR = /cluster/scratch/gguidarini/protein-design/train
60 of 60 baseline runs missing — run the launcher(s) above. First few:
   lowdata_vanilla_35m_n20_s0  [ed1_m22]
   lowdata_vanilla_35m_n20_s1  [ed1_m22]
   lowdata_vanilla_35m_n20_s2  [ed1_m22]
   lowdata_vanilla_35m_n50_s0  [ed1_m22]
   lowdata_vanilla_35m_n50_s1  [ed1_m22]
   lowdata_vanilla_35m_n50_s2  [ed1_m22]
   lowdata_vanilla_35m_n100_s0  [

In [4]:
# === A2: baseline learning-curve plotter (14-07-26 _plot_lc, trimmed) ========
def plot_baseline_lc(models, test_set, df, metric=_METRIC, title="", save_path=None, total_train=None):
    fin = df[(df["test_set"] == test_set) & pd.to_numeric(df[metric], errors="coerce").notna()].copy()
    if fin.empty:
        print(f"[{test_set}] nothing to plot yet — run the baseline launcher above.")
        return None
    fin[metric] = pd.to_numeric(fin[metric])
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, mk in enumerate(models):
        sub = fin[fin["model"] == mk]
        if sub.empty:
            continue
        agg = (sub.groupby("n_train")[metric].agg(mean="mean", std="std", n="size")
                  .reset_index().sort_values("n_train"))
        color = lc_color(mk, i)
        ax.plot(agg["n_train"], agg["mean"], marker="o", color=color, label=lc_label(mk))
        std = agg["std"].fillna(0.0)
        ax.fill_between(agg["n_train"], agg["mean"] - std, agg["mean"] + std, color=color, alpha=0.12)
    ax.set_xscale("log")
    ax.set_xlabel("# training sequences (N)")
    ax.set_ylabel("Spearman (single held-out test split)")
    ax.set_title(title)
    ax.axhline(0.0, color="0.7", lw=0.8, ls=":", zorder=0)
    ax.grid(True, which="both", alpha=0.2)
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    if total_train:
        secax = ax.secondary_xaxis("top", functions=(
            lambda x: 100 * x / total_train, lambda p: p * total_train / 100))
        secax.set_xlabel(f"% of {total_train}-row train split")
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight"); print("Saved:", save_path)
    return fig

for key, spec in DATASETS.items():
    fig = plot_baseline_lc(
        MODELS, spec["test_set"], baseline_df,
        title=f"[Baseline / non-CV] DPO low-data curve — {spec['test_set']}",
        save_path=str(FIG_DIR / f"baseline_noncv_curve_{spec['test_set']}_{DATE}.pdf"),
        total_train=spec["total_train"],
    )
    if fig is not None:
        plt.show()

[ed1] nothing to plot yet — run the baseline launcher above.
[cetuximab_h] nothing to plot yet — run the baseline launcher above.


## Section B — k-fold CV setup & artifact preflight

**Method.** For a dataset `D = {(x_j, y_j)}` of size `n`:

1. deterministically assign each row to one of `k` folds (metric-stratified,
   seeded by `fold_seed` — independent of the fold index, so every fold sees the
   same assignment);
2. for fold `i`, **fold `i` is the held-out test set**;
3. the remaining `k-1` folds are the non-test pool, split into train/val
   (**val is drawn only from the pool — the held-out fold never tunes anything**);
4. run the *same* low-data DPO pipeline on that fold-specific split;
5. export fold `i`'s held-out `(prediction, ground_truth)` pairs
   (`test_predictions.csv`);
6. after all `k` folds finish, **concatenate** all out-of-fold predictions and
   compute **one pooled Spearman** over the full dataset.

That pooled OOF Spearman (n≈456 for ED1, 559 for Cetuximab — the *whole*
dataset, vs a single ~44–56-row split) is the headline metric, and it is **not**
the mean of per-fold Spearmans.

All of this is implemented once in the reusable library / config / scripts, so
this notebook only *reads* the artifacts:

- fold splits: `protein_design.cv_splitting` (+ `conf/data/dms/cv.yaml` for the
  `cv:` block: `n_folds`, `fold_seed`, `val_frac`, `output_dir`);
- launch: `bash_scripts/dpo_cv_sweep.sh` (pre-builds fold splits, then submits
  one job per `(model, N, seed, fold)` reusing `dpo_lowdata_sweep.sh`);
- pooling: `protein_design.cv_splitting.pooled_oof_spearman` +
  `scripts/analysis/aggregate_cv_folds.py`.

In [5]:
# === B0: import the reusable CV library (keeps the notebook thin) ===========
from protein_design.cv_splitting import (
    cv_fold_key, parse_cv_fold_key, pooled_oof_spearman,
)

# The commands needed to (1) materialize fold splits and (2) launch the CV
# sweep. Printed for the user to run — this notebook never submits jobs.
print("# --- 1) pre-materialize the k-fold splits (sequential, no job races) ---")
print(f"uv run python scripts/data_prep/build_cv_splits.py --dms-config conf/data/dms/cv.yaml \\")
print(f"    --datasets {','.join(DATASETS)} --n-folds {N_FOLDS} --fold-seed {FOLD_SEED} --verify\n")
print("# --- 2) launch the CV low-data sweep (one call per dataset) ---")
for key, spec in DATASETS.items():
    ng = ",".join(str(n) for n in spec["n_grid"])
    print(f"bash_scripts/dpo_cv_sweep.sh --dataset {key} --models {','.join(MODELS)} "
          f"--n {ng} --seeds {','.join(map(str, SEEDS))} --n-folds {N_FOLDS} --fold-seed {FOLD_SEED}")

# --- 1) pre-materialize the k-fold splits (sequential, no job races) ---
uv run python scripts/data_prep/build_cv_splits.py --dms-config conf/data/dms/cv.yaml \
    --datasets ed1_m22,cetuximab_h --n-folds 5 --fold-seed 0 --verify

# --- 2) launch the CV low-data sweep (one call per dataset) ---
bash_scripts/dpo_cv_sweep.sh --dataset ed1_m22 --models vanilla_35m,evo_35m --n 20,50,100,200,366 --seeds 0,1,2 --n-folds 5 --fold-seed 0
bash_scripts/dpo_cv_sweep.sh --dataset cetuximab_h --models vanilla_35m,evo_35m --n 20,50,100,200,447 --seeds 0,1,2 --n-folds 5 --fold-seed 0


In [ ]:
# === B1: CV artifact preflight — fold splits + fold runs =====================
# (a) fold splits present?
print("Fold splits under", CV_SPLIT_DIR, ":")
splits_ok = True
for key in DATASETS:
    have = [i for i in range(N_FOLDS)
            if (CV_SPLIT_DIR / cv_fold_key(key, N_FOLDS, FOLD_SEED, i) / "test.csv").exists()]
    print(f"  {key}: {len(have)}/{N_FOLDS} folds materialized")
    splits_ok = splits_ok and len(have) == N_FOLDS
if not splits_ok:
    print("  -> missing fold splits: run the build_cv_splits.py command from B0.")

# (b) fold RUNS present in TRAIN_DIR? Count finished fold runs (summary.json +
# test_predictions.csv) per (dataset, model, N, seed); a cell is complete when
# all N_FOLDS folds are finished.
# The run NAME only carries the _cv<K>s<FS>_f<I> suffix (base_name suffix from
# dpo_cv_sweep.sh) — NOT the base dataset, and ed1_m22 vs cetuximab_h share the
# same suffix. So the base dataset is read authoritatively from each run's
# resolved_config.yaml (data.test.dataset_key -> parse_cv_fold_key).
_CV_RE = re.compile(
    rf"^lowdata_(?P<model>{_MODELS_RE})_n(?P<n>\d+)_s(?P<seed>\d+)_cv\d+s\d+_f\d+$")

def scan_cv_runs(train_dir=TRAIN_DIR) -> pd.DataFrame:
    cols = ["model", "base_dataset", "n_train", "seed", "n_folds", "fold_seed", "fold", "run_dir"]
    if not train_dir.exists():
        return pd.DataFrame(columns=cols)
    best = {}
    for p in sorted(train_dir.glob("lowdata_*_cv*_f*")):
        name = _TS_RE.sub("", p.name)
        m = _CV_RE.match(name)
        if not m:
            continue
        if not (p / "summary.json").exists() or not (p / "test_predictions.csv").exists():
            continue
        dkey = _run_test_dataset_key(p)
        parsed = parse_cv_fold_key(dkey) if dkey else None
        if parsed is None:
            continue
        key = (m.group("model"), parsed.base_key, int(m.group("n")), int(m.group("seed")),
               parsed.n_folds, parsed.fold_seed, parsed.fold_index)
        try:
            mt = (p / "summary.json").stat().st_mtime
        except OSError:
            continue
        if key in best and best[key][0] >= mt:
            continue
        best[key] = (mt, str(p))
    rows = [{"model": k[0], "base_dataset": k[1], "n_train": k[2], "seed": k[3],
             "n_folds": k[4], "fold_seed": k[5], "fold": k[6], "run_dir": v[1]}
            for k, v in best.items()]
    if not rows:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows).sort_values(
        ["base_dataset", "model", "n_train", "seed", "fold"]).reset_index(drop=True)

cv_runs_df = scan_cv_runs()
print(f"\nFinished CV fold runs found: {len(cv_runs_df)} "
      f"(expected {sum(len(MODELS)*len(s['n_grid'])*len(SEEDS)*N_FOLDS for s in DATASETS.values())})")
if not cv_runs_df.empty:
    complete = (cv_runs_df.groupby(["base_dataset", "model", "n_train", "seed"])["fold"]
                .nunique().rename("folds_done").reset_index())
    n_complete = int((complete["folds_done"] == N_FOLDS).sum())
    print(f"(model,N,seed) cells with all {N_FOLDS} folds finished: {n_complete} / {len(complete)}")

## Section C — CV learning curves (pooled out-of-fold Spearman)

For each `(model, N, low_data_seed)` we concatenate the held-out predictions
from all `k` folds and compute **one** Spearman over the pooled full dataset
(`pooled_oof_spearman`). We then aggregate across low-data seeds exactly as the
baseline does — mean ± std of the *per-seed pooled score* — so the only thing
that changed versus Section A is the **evaluation** (pooled OOF instead of a
single fixed split), not the aggregation.

Cells that lack all `k` folds are skipped cleanly (and reported in Section E).

In [ ]:
# === C0: pool per (dataset, model, N, seed); aggregate across seeds =========
def build_cv_pooled_df(cv_runs, datasets=DATASETS, k=N_FOLDS):
    # One pooled OOF Spearman per (base_dataset, model, N, seed) with all k folds.
    cols = ["base_dataset", "model", "n_train", "seed", "pooled_spearman", "n_pooled",
            "folds_used", "foldwise"]
    if cv_runs.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    grp = cv_runs.groupby(["base_dataset", "model", "n_train", "seed"])
    for (base, model, n, seed), g in grp:
        if base not in datasets:
            continue
        if g["fold"].nunique() < k:
            continue  # incomplete cell — reported in Section E
        fold_dfs = []
        for run_dir in g.sort_values("fold")["run_dir"]:
            try:
                fold_dfs.append(pd.read_csv(Path(run_dir) / "test_predictions.csv"))
            except Exception:
                pass
        if len(fold_dfs) < k:
            continue
        pooled = pooled_oof_spearman(fold_dfs, pred_col="score",
                                     truth_col="M22_binding_enrichment_adj")
        rows.append({"base_dataset": base, "model": model, "n_train": int(n), "seed": int(seed),
                     "pooled_spearman": pooled["pooled_spearman"], "n_pooled": pooled["n_pooled"],
                     "folds_used": pooled["n_folds"], "foldwise": pooled["foldwise_spearman"]})
    if not rows:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows).sort_values(["base_dataset", "model", "n_train", "seed"]).reset_index(drop=True)

cv_pooled_df = build_cv_pooled_df(cv_runs_df)
print(f"Pooled-CV points (one per model,N,seed with all {N_FOLDS} folds): {len(cv_pooled_df)}")
if not cv_pooled_df.empty:
    print(cv_pooled_df.drop(columns=["foldwise"]).to_string(index=False))

In [ ]:
# === C1: CV learning-curve plot (pooled OOF Spearman, mean +/- std / seeds) ==
def plot_cv_lc(models, base_dataset, pooled_df, title="", save_path=None, total_train=None):
    fin = pooled_df[(pooled_df["base_dataset"] == base_dataset)
                    & pd.to_numeric(pooled_df["pooled_spearman"], errors="coerce").notna()].copy()
    if fin.empty:
        print(f"[{base_dataset}] no complete CV cells yet — run the CV sweep (Section B).")
        return None
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, mk in enumerate(models):
        sub = fin[fin["model"] == mk]
        if sub.empty:
            continue
        agg = (sub.groupby("n_train")["pooled_spearman"].agg(mean="mean", std="std", n="size")
                  .reset_index().sort_values("n_train"))
        color = lc_color(mk, i)
        ax.plot(agg["n_train"], agg["mean"], marker="o", color=color, label=lc_label(mk))
        std = agg["std"].fillna(0.0)
        ax.fill_between(agg["n_train"], agg["mean"] - std, agg["mean"] + std, color=color, alpha=0.12)
    ax.set_xscale("log")
    ax.set_xlabel("# training sequences (N)")
    ax.set_ylabel(f"Pooled out-of-fold Spearman ({N_FOLDS}-fold, full dataset)")
    ax.set_title(title)
    ax.axhline(0.0, color="0.7", lw=0.8, ls=":", zorder=0)
    ax.grid(True, which="both", alpha=0.2)
    ax.legend(frameon=False, fontsize=9, loc="upper left")
    if total_train:
        secax = ax.secondary_xaxis("top", functions=(
            lambda x: 100 * x / total_train, lambda p: p * total_train / 100))
        secax.set_xlabel(f"% of {total_train}-row train pool")
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight"); print("Saved:", save_path)
    return fig

for key, spec in DATASETS.items():
    fig = plot_cv_lc(
        MODELS, key, cv_pooled_df,
        title=f"[{N_FOLDS}-fold CV] pooled-OOF DPO low-data curve — {spec['test_set']}",
        save_path=str(FIG_DIR / f"cv_pooled_curve_{spec['test_set']}_{DATE}.pdf"),
        total_train=spec["total_train"],
    )
    if fig is not None:
        plt.show()

## Section D — comparison: baseline vs pooled-CV (variance reduction)

The point of CV here is **not** to change the evo-vs-vanilla story but to
**shrink the uncertainty band**. Two comparisons, kept in separate figures (per
the brief — never one merged plot):

1. **Side-by-side curves** — baseline (single split) vs pooled-CV, one panel
   each, shared y-axis, per dataset.
2. **Seed-to-seed std vs N** — the width of the uncertainty band itself
   (std across the 3 low-data seeds) for baseline vs CV. If CV reduces test-set
   noise, its band is narrower, especially at low N.

In [ ]:
# === D1: side-by-side baseline vs CV curves (separate panels) ================
def _agg_mean_std(df, value_col, model):
    sub = df[df["model"] == model]
    if sub.empty:
        return None
    return (sub.groupby("n_train")[value_col].agg(mean="mean", std="std", n="size")
               .reset_index().sort_values("n_train"))

for key, spec in DATASETS.items():
    base_sub = baseline_df[baseline_df["test_set"] == spec["test_set"]].copy()
    base_sub[_METRIC] = pd.to_numeric(base_sub[_METRIC], errors="coerce")
    base_sub = base_sub[base_sub[_METRIC].notna()]
    cv_sub = cv_pooled_df[cv_pooled_df["base_dataset"] == key]
    if base_sub.empty and cv_sub.empty:
        print(f"[{spec['test_set']}] neither baseline nor CV data yet — skipping comparison.")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    panels = [("Baseline (single split)", base_sub, _METRIC, axes[0]),
              (f"{N_FOLDS}-fold pooled OOF", cv_sub, "pooled_spearman", axes[1])]
    for ptitle, pdf, vcol, ax in panels:
        for i, mk in enumerate(MODELS):
            agg = _agg_mean_std(pdf, vcol, mk) if not pdf.empty else None
            if agg is None or agg.empty:
                continue
            color = lc_color(mk, i)
            ax.plot(agg["n_train"], agg["mean"], marker="o", color=color, label=lc_label(mk))
            std = agg["std"].fillna(0.0)
            ax.fill_between(agg["n_train"], agg["mean"] - std, agg["mean"] + std, color=color, alpha=0.12)
        ax.set_xscale("log")
        ax.set_xlabel("# training sequences (N)")
        ax.set_title(ptitle)
        ax.axhline(0.0, color="0.7", lw=0.8, ls=":", zorder=0)
        ax.grid(True, which="both", alpha=0.2)
        ax.legend(frameon=False, fontsize=9, loc="upper left")
    axes[0].set_ylabel("Spearman (held-out)")
    fig.suptitle(f"Baseline vs {N_FOLDS}-fold CV — {spec['test_set']}")
    fig.tight_layout()
    sp = FIG_DIR / f"baseline_vs_cv_{spec['test_set']}_{DATE}.pdf"
    fig.savefig(sp, bbox_inches="tight"); print("Saved:", sp)
    plt.show()

In [ ]:
# === D2: seed-to-seed std vs N — is the CV band narrower? =====================
def _std_by_n(df, value_col, model):
    sub = df[df["model"] == model]
    if sub.empty:
        return None
    g = (sub.groupby("n_train")[value_col].agg(std="std", n="size").reset_index().sort_values("n_train"))
    return g[g["n"] >= 2]  # std needs >=2 seeds

for key, spec in DATASETS.items():
    base_sub = baseline_df[baseline_df["test_set"] == spec["test_set"]].copy()
    base_sub[_METRIC] = pd.to_numeric(base_sub[_METRIC], errors="coerce")
    cv_sub = cv_pooled_df[cv_pooled_df["base_dataset"] == key]
    if base_sub.empty and cv_sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 5))
    any_data = False
    for i, mk in enumerate(MODELS):
        color = lc_color(mk, i)
        b = _std_by_n(base_sub, _METRIC, mk)
        c = _std_by_n(cv_sub, "pooled_spearman", mk)
        if b is not None and not b.empty:
            ax.plot(b["n_train"], b["std"], marker="o", ls="--", color=color,
                    label=f"{lc_label(mk)} — baseline"); any_data = True
        if c is not None and not c.empty:
            ax.plot(c["n_train"], c["std"], marker="s", ls="-", color=color,
                    label=f"{lc_label(mk)} — CV"); any_data = True
    if not any_data:
        plt.close(fig)
        print(f"[{spec['test_set']}] need >=2 seeds in both to compare std — skipping.")
        continue
    ax.set_xscale("log")
    ax.set_xlabel("# training sequences (N)")
    ax.set_ylabel("std of Spearman across low-data seeds")
    ax.set_title(f"Seed-to-seed variance — baseline (dashed) vs CV (solid) — {spec['test_set']}")
    ax.grid(True, which="both", alpha=0.2)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    sp = FIG_DIR / f"cv_variance_reduction_{spec['test_set']}_{DATE}.pdf"
    fig.savefig(sp, bbox_inches="tight"); print("Saved:", sp)
    plt.show()

## Section E — diagnostics

Sanity checks for the CV run: fold sizes, pooled example counts, fold
completion status, foldwise Spearmans (diagnostics only — **not** the headline
metric), and a seed-to-seed variance summary.

In [ ]:
# === E1: rows per fold (from the materialized split CSVs) ====================
rows = []
for key in DATASETS:
    for i in range(N_FOLDS):
        fk = cv_fold_key(key, N_FOLDS, FOLD_SEED, i)
        d = CV_SPLIT_DIR / fk
        rec = {"dataset": key, "fold": i}
        for split in ("train", "val", "test"):
            fp = d / f"{split}.csv"
            rec[split] = (sum(1 for _ in open(fp)) - 1) if fp.exists() else None
        rows.append(rec)
fold_sizes = pd.DataFrame(rows, columns=["dataset", "fold", "train", "val", "test"])
print("Rows per fold (materialized splits):")
print(fold_sizes.to_string(index=False))
if fold_sizes["test"].notna().any():
    tot = fold_sizes.dropna(subset=["test"]).groupby("dataset")["test"].sum()
    print("\nPooled held-out rows per dataset (sum of fold test sizes = full dataset):")
    print(tot.to_string())

In [ ]:
# === E2: fold completion matrix (which (model,N,seed,fold) runs finished) ====
if cv_runs_df.empty:
    print("No CV fold runs found yet — nothing to report (see Section B for the launch command).")
else:
    comp = (cv_runs_df.groupby(["base_dataset", "model", "n_train", "seed"])["fold"]
            .nunique().rename("folds_done").reset_index())
    comp["complete"] = comp["folds_done"] == N_FOLDS
    print(f"Cells complete ({N_FOLDS}/{N_FOLDS} folds): {int(comp['complete'].sum())} / {len(comp)}")
    incomplete = comp[~comp["complete"]]
    if not incomplete.empty:
        print("\nIncomplete cells (need a rerun of the missing folds):")
        print(incomplete.to_string(index=False))

In [ ]:
# === E3: foldwise Spearman table (diagnostics only) + pooled comparison ======
if cv_pooled_df.empty:
    print("No complete pooled-CV cells yet.")
else:
    diag = cv_pooled_df.copy()
    diag["mean_foldwise"] = diag["foldwise"].apply(
        lambda xs: float(np.nanmean(xs)) if xs else float("nan"))
    diag["std_foldwise"] = diag["foldwise"].apply(
        lambda xs: float(np.nanstd(xs)) if xs and len(xs) > 1 else float("nan"))
    show = diag[["base_dataset", "model", "n_train", "seed", "pooled_spearman",
                 "mean_foldwise", "std_foldwise", "n_pooled"]]
    print("Pooled OOF (headline) vs mean-of-foldwise (diagnostic only):")
    print(show.round(4).to_string(index=False))
    print("\nNOTE: 'pooled_spearman' is the headline metric (one Spearman over all "
          "pooled OOF pairs). 'mean_foldwise' is shown only to illustrate they differ.")

In [ ]:
# === E4: seed-to-seed variance summary — baseline vs CV ======================
summ = []
for key, spec in DATASETS.items():
    b = baseline_df[baseline_df["test_set"] == spec["test_set"]].copy()
    b[_METRIC] = pd.to_numeric(b[_METRIC], errors="coerce")
    c = cv_pooled_df[cv_pooled_df["base_dataset"] == key]
    for mk in MODELS:
        bstd = (b[b["model"] == mk].groupby("n_train")[_METRIC].std())
        cstd = (c[c["model"] == mk].groupby("n_train")["pooled_spearman"].std())
        summ.append({
            "dataset": spec["test_set"], "model": mk,
            "baseline_mean_std": float(np.nanmean(bstd.values)) if len(bstd) else float("nan"),
            "cv_mean_std": float(np.nanmean(cstd.values)) if len(cstd) else float("nan"),
        })
var_summary = pd.DataFrame(summ, columns=["dataset", "model", "baseline_mean_std", "cv_mean_std"])
var_summary["cv_reduces_std"] = var_summary["cv_mean_std"] < var_summary["baseline_mean_std"]
print("Mean (over N) of the seed-to-seed std — lower = more stable curve:")
print(var_summary.round(4).to_string(index=False))

## Conclusion

**What this notebook establishes (once the CV sweep has run):**

- **Baseline (Section A)** reproduces the 14-07-26 single-split low-data curves
  unchanged — the reference, kept visually separate from every CV result.
- **k-fold CV (Sections C–D)** replaces the single ~44–56-row held-out split
  with a **pooled out-of-fold Spearman over the entire dataset** (n≈456 ED1 /
  559 Cetuximab). Because every row is scored exactly once out-of-fold and the
  pooled evaluation uses `k×` more test points, the seed-to-seed band (Section
  D2 / E4) should be **narrower than the baseline**, especially at low N where
  the single-split estimate was noisiest — that is the variance-reduction claim.
- CV is a change to **evaluation only**: identical datasets, low-data
  subsampling, DPO/LoRA-DPO machinery, and seed aggregation. It is introduced as
  a *separate method*, never as a silent replacement of the baseline.

**Correctness guarantees (enforced in the library, tested in
`tests/test_cv_splitting.py`):** pooled Spearman is computed on concatenated OOF
predictions (not a mean of per-fold rho); the held-out fold never leaks into
train or val (val is drawn only from the remaining folds); fold assignment is
deterministic in `(n_folds, fold_seed)`; and each row is held out exactly once.

**Reading the plots before the sweep finishes:** every section degrades
gracefully — missing artifacts print the exact `uv run` / `bash_scripts` command
to build them and the corresponding plot is skipped rather than crashing.

**Next variance-reduction methods** (future sections, same modular structure):
repeated k-fold with multiple `fold_seed`s, or a nested-CV band — each added as
its own Section, with the baseline kept first for comparison.